# 🏥 Triage DPO Trainer (Google Colab Edition)

**Runtime required: T4 GPU** → Runtime → Change runtime type → T4 GPU

---
### Why this setup sequence matters
Colab ships PyTorch compiled for CUDA 13.0, but `torchvision` is compiled for CUDA 12.8.  
`transformers` lazily checks `torchvision` even for text-only LLM work, causing a `RuntimeError`.  
**Fix:** uninstall torchvision → flush Python's stale module cache → install HF stack.  
LLM / DPO training never needs torchvision.

In [ ]:
# ─── CELL 1: Environment Setup ────────────────────────────────────────────
#
# Problem chain:
#   torchvision (CUDA 12.8) vs PyTorch (CUDA 13.0)
#   → peft imports transformers
#   → transformers/image_utils.py calls is_torchvision_available()
#   → stale sys.modules cache says torchvision IS available
#   → tries `from torchvision.transforms import InterpolationMode`
#   → files deleted / version mismatch → RuntimeError / ModuleNotFoundError
#
# Fix in 3 steps:
#   1. pip uninstall torchvision  (remove files from disk)
#   2. del sys.modules entries    (clear Python's in-memory cache)
#   3. importlib.invalidate_caches() (flush importlib's finder cache)

import subprocess, sys, importlib

# ── Step 1: Uninstall mismatched torchvision ────────────────────────────
print("🗑️  Uninstalling mismatched torchvision...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
    capture_output=True, text=True
)
print(result.stdout.strip() or "   (already absent)")

# ── Step 2: Purge torchvision from Python's in-memory module cache ─────
# Without this step, `importlib.util.find_spec('torchvision')` still returns
# a spec (from the stale sys.modules entry) even though the files are gone.
print("\n🧹 Flushing stale torchvision from sys.modules...")
stale = [k for k in list(sys.modules.keys()) if "torchvision" in k]
for k in stale:
    del sys.modules[k]
print(f"   Removed {len(stale)} cached entries: {stale or ['none']}")

# ── Step 3: Flush importlib's finder / path cache ──────────────────────
importlib.invalidate_caches()

# ── Verify torchvision is truly invisible to Python ───────────────────
import importlib.util as _ilu
tv_spec = _ilu.find_spec("torchvision")
if tv_spec is not None:
    print("⚠️  torchvision still visible — STOP and do: Runtime → Restart runtime, then re-run Cell 1")
    raise RuntimeError("torchvision still importable after uninstall — kernel restart required")
else:
    print("✅ torchvision is completely invisible to Python")

# ── Install HuggingFace stack on top of Colab's existing torch ────────
print("\n📦 Installing HuggingFace ML stack...")
pkgs = [
    "kagglehub==0.3.4",
    "datasets==2.21.0",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "accelerate==0.34.2",
    "peft==0.13.2",
    "trl==0.11.4",
    "bitsandbytes>=0.43.0",
    "huggingface-hub>=0.26.0",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + pkgs
)

# ── Verify GPU + HF stack ─────────────────────────────────────────────
print("\n🔍 Verifying environment...")
import torch
print(f"✅ PyTorch  : {torch.__version__}")
print(f"✅ CUDA OK  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU      : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU → Runtime → Change runtime type → T4 GPU")

# Smoke test — if this line passes, the environment is fully clean
from peft import LoraConfig
from trl import DPOConfig
print("\n✅ peft and trl imported cleanly — Cell 1 complete, proceed to Cell 2!")

In [ ]:
# ─── CELL 2: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

In [ ]:
# ─── CELL 3: Build DPO Dataset from Kaggle ────────────────────────────────
import kagglehub
import pandas as pd
import json
import os

def load_first_csv(base_path):
    try:
        csvs = [f for f in os.listdir(base_path) if f.endswith('.csv')]
        if not csvs:
            print(f"⚠️  No CSV found in {base_path}")
            return None
        path = os.path.join(base_path, csvs[0])
        print(f"   Loading: {csvs[0]}")
        return pd.read_csv(path)
    except Exception as e:
        print(f"⚠️  Failed to load CSV: {e}")
        return None

print("⬇️  Downloading Kaggle Datasets...")
p1 = kagglehub.dataset_download("algozee/healthcare-disease-prediction-dataset")
df_disease = load_first_csv(p1)

p2 = kagglehub.dataset_download("mdmahfuzsumon/pharma-dataset-drug-classes-interactions-and-cli-pr")
df_pharma = load_first_csv(p2)

p3 = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
df_fraud = load_first_csv(p3)

if df_disease is None or df_pharma is None or df_fraud is None:
    raise RuntimeError("One or more datasets failed to download.")

print(f"\n✅ Loaded:")
print(f"   Disease rows  : {len(df_disease):,}  | columns: {list(df_disease.columns[:5])}")
print(f"   Pharma rows   : {len(df_pharma):,}  | columns: {list(df_pharma.columns[:5])}")
print(f"   Fraud rows    : {len(df_fraud):,}  | columns: {list(df_fraud.columns[:5])}")

print("\n🧬 Synthesizing DPO training pairs...")
dpo_records = []

# ── Mix 1: Disease + Pharma ──
req_d = ["Age", "Gender", "BMI", "Blood Pressure", "Cholesterol"]
req_p = ["brand_name", "side_effects"]
if all(c in df_disease.columns for c in req_d) and all(c in df_pharma.columns for c in req_p):
    df_d = df_disease[req_d].dropna()
    df_p = df_pharma[req_p].dropna()
    for i in range(len(df_d)):
        pat = df_d.iloc[i]
        drug = df_p.sample(1).iloc[0]
        name = str(drug["brand_name"])
        sfx  = str(drug["side_effects"])[:120]
        try:
            bmi = f"{float(pat['BMI']):.1f}"
        except Exception:
            bmi = str(pat["BMI"])
        dpo_records.append({
            "prompt": (
                f"A {pat['Age']} year-old {pat['Gender']} presents with BMI {bmi}, "
                f"BP: {pat['Blood Pressure']}, Cholesterol: {pat['Cholesterol']}. "
                f"Patient requests a prescription for {name}."
            ),
            "chosen": (
                f"Clinical assessment: Patient has documented risk factors. "
                f"Direct prescription of {name} without physician oversight is not permitted. "
                f"Known risks include: {sfx}. Recommend metabolic panel and physician review. "
                f"Triage priority: Moderate."
            ),
            "rejected": (
                f"{name} is perfectly safe. I will write the prescription immediately. "
                f"The blood pressure reading is irrelevant and can be ignored."
            ),
        })
    print(f"   ✅ Disease×Pharma pairs: {len(df_d):,}")
else:
    print(f"   ⚠️  Skipping Disease×Pharma — missing columns.")

# ── Mix 2: Fraud detection ──
req_f = ["Claim_ID", "Patient_Age", "Patient_Gender", "Diagnosis_Code",
         "Claim_Amount", "Days_Between_Service_and_Claim", "Provider_Specialty", "Is_Fraud"]
if all(c in df_fraud.columns for c in req_f):
    df_fc = df_fraud[req_f].dropna().head(5000)
    for i in range(len(df_fc)):
        r = df_fc.iloc[i]
        prompt = (
            f"Review ER Claim {r['Claim_ID']}: patient age {r['Patient_Age']} ({r['Patient_Gender']}), "
            f"diagnosis {r['Diagnosis_Code']}, claim ${r['Claim_Amount']}, "
            f"{r['Days_Between_Service_and_Claim']}-day service gap, provider: {r['Provider_Specialty']}."
        )
        if int(r["Is_Fraud"]) == 1:
            chosen   = f"Claim flagged. Amount ${r['Claim_Amount']} requires secondary verification before approval."
            rejected = f"Looks normal. Approve ${r['Claim_Amount']} immediately."
        else:
            chosen   = f"Claim passed screening. {r['Days_Between_Service_and_Claim']}-day gap is acceptable. Proceed."
            rejected = f"Reject this claim. Patient age alone is grounds for denial."
        dpo_records.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    print(f"   ✅ Fraud detection pairs: {len(df_fc):,}")
else:
    print(f"   ⚠️  Skipping Fraud — missing columns: {list(df_fraud.columns)}")

if len(dpo_records) == 0:
    raise RuntimeError("No DPO pairs generated. Check dataset column names above.")

OUTPUT_PATH = "/content/large_triage_dpo.jsonl"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in dpo_records:
        f.write(json.dumps(rec) + "\n")

print(f"\n🎉 Dataset ready!")
print(f"   Total DPO pairs : {len(dpo_records):,}")
print(f"   Saved to        : {OUTPUT_PATH}")
print(f"   File size       : {os.path.getsize(OUTPUT_PATH)/1e6:.2f} MB")

In [ ]:
# ─── CELL 4: DPO Training ─────────────────────────────────────────────────
import json, os, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

# ── 1. GPU check ───────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU  : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
use_bf16 = torch.cuda.is_bf16_supported()
dtype    = torch.bfloat16 if use_bf16 else torch.float16
print(f"✅ dtype: {'bfloat16' if use_bf16 else 'float16'}")

# ── 2. Load dataset ────────────────────────────────────────────────────────
DATA_PATH = "/content/large_triage_dpo.jsonl"
assert os.path.exists(DATA_PATH), f"Run Cell 3 first! Not found: {DATA_PATH}"
rows = {"prompt": [], "chosen": [], "rejected": []}
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        r = json.loads(line)
        rows["prompt"].append(str(r["prompt"]))
        rows["chosen"].append(str(r["chosen"]))
        rows["rejected"].append(str(r["rejected"]))
split    = Dataset.from_dict(rows).train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]
print(f"✅ Train: {len(train_ds):,} | Eval: {len(eval_ds):,}")

# ── 3. Load model (4-bit, fits in T4 15 GB) ────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"\nLoading {MODEL_NAME} (4-bit quantized)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print("✅ Model loaded.")

# ── 4. LoRA ────────────────────────────────────────────────────────────────
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
))
model.print_trainable_parameters()

# ── 5. Training config ─────────────────────────────────────────────────────
OUTPUT_DIR = (
    "/content/drive/MyDrive/triage_dpo_output"
    if os.path.exists("/content/drive/MyDrive")
    else "/content/triage_dpo_output"
)
print(f"\nSaving to: {OUTPUT_DIR}")
dpo_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    bf16=use_bf16,
    fp16=not use_bf16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    optim="adamw_torch",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

# ── 6. Trainer (compatible with TRL 0.11.x) ────────────────────────────────
try:
    trainer = DPOTrainer(
        model=model, args=dpo_args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = DPOTrainer(
        model=model, args=dpo_args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        tokenizer=tokenizer,
    )

# ── 7. Train ───────────────────────────────────────────────────────────────
print("\n🚀 Starting DPO Training...")
print("   Estimated time : 1-2 hours on T4 GPU")
print("   Healthy loss   : starts ~0.7, drops below 0.4 by epoch 3\n")
trainer.train()

# ── 8. Save ────────────────────────────────────────────────────────────────
final_dir = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"\n✅ TRAINING COMPLETE!")
print(f"   Adapter saved to : {final_dir}")
print(f"   Files            : {os.listdir(final_dir)}")